![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 24 -- Lab 2: Brain MRI with a U-Net

A segmentation task from Kaggle: 3,064 brain MRI slices, each with a mask of the tumour drawn by radiologists, scored by Dice. You write a Dataset that resizes image and mask together and flips them together, build the deck's U-Net as a class with its skip connections, train it from scratch with a weighted BCE loss, read the Dice curve, test the best epoch once with Dice, IoU and pixel accuracy, look at the predicted masks, and save them as the hand-in.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Kaggle `nikhilroxtomar/brain-tumor-segmentation` (about 330 MB; the slices of Cheng et al., 2015), downloaded by `kagglehub` in the setup cell: `images/N.png` and `masks/N.png`, 512 x 512 greyscale, mask pixels 0 or 255. You use 1,600 of the slices at 128 x 128: 1,200 training, 200 validation, 200 test. A GPU makes each epoch a few seconds; on a CPU about a minute.

---
# Setup

In [ ]:
# kagglehub, torch, torchvision, matplotlib, pandas and numpy are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q kagglehub torch torchvision matplotlib pandas numpy

In [ ]:
import os
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import to_tensor
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

In [ ]:
path = kagglehub.dataset_download("nikhilroxtomar/brain-tumor-segmentation")
IMG, MASK = os.path.join(path, "images"), os.path.join(path, "masks")
names = sorted(os.listdir(IMG), key=lambda s: int(s[:-4]))
perm = torch.randperm(len(names), generator=torch.Generator().manual_seed(0)).tolist()
train_names = [names[k] for k in perm[:1200]]
val_names = [names[k] for k in perm[1200:1400]]
test_names = [names[k] for k in perm[1400:1600]]
print(len(names), "slices; using", len(train_names), len(val_names), len(test_names))

---
# Part 1 -- Images and masks, together

The *A Mask Is a Grid of Class Numbers* slide. Whatever is done to the image must be done to the mask: same resize, same flip.

## Task 1: Look at the data

Show 6 training slices with their masks drawn on top: `plt.imshow(image, cmap="gray")` then `plt.imshow(mask, cmap="Reds", alpha=0.4)`, where both come from `Image.open(...)`. Print the size and mode of one image and one mask, the unique values of the mask (`np.unique`), and the share of tumour pixels in the first 200 masks (mean of `(mask > 0).mean()`).

In [ ]:
# Your code here


## Task 2: The Dataset class

Write `class TumourDataset(Dataset)` with `__init__(self, names, size=128, augment=False)` that caches, for every name, the image `to_tensor(Image.open(...).convert("L").resize((size, size)))` (shape `(1, 128, 128)`, values 0 to 1) and the mask `to_tensor(Image.open(...).convert("L").resize((size, size), Image.NEAREST))` (values 0 or 1, nearest so no grey appears); `__len__`; and `__getitem__(self, i)` that returns `(x, y)`, flipping **both** left-right with `x.flip(-1), y.flip(-1)` with probability 0.5 when `augment` is on. Build `train_ds = TumourDataset(train_names, augment=True)`, `val_ds` and `test_ds` without augmentation; print `len(train_ds)` and the shapes and value sets of `train_ds[0]`.

Syntax hint (the shape of the call, not the answer):

```python
class MyData(Dataset):
    def __init__(self, names, ...):   # decode and resize once, keep tensors in lists
    def __len__(self):
    def __getitem__(self, i):          # return (image_i, mask_i), transformed together
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(train_ds) == 1200 and tuple(x0.shape) == (1, 128, 128) and tuple(y0.shape) == (1, 128, 128) and set(torch.unique(y0).tolist()) <= {0.0, 1.0}, 'TumourDataset: (1, 128, 128) image in 0..1 and a 0/1 mask of the same shape; NEAREST for the mask'
print('Task 2 passed')

## Task 3: Loaders

Build `train_loader` (batch 16, `shuffle=True`), `val_loader` and `test_loader` (batch 32). Take one training batch and print the two shapes: `(16, 1, 128, 128)` twice.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(xb.shape) == (16, 1, 128, 128) and tuple(yb.shape) == (16, 1, 128, 128), 'loaders: batches of 16 image and mask tensors'
print('Task 3 passed')

---
# Part 2 -- The U-Net class

The *In Code: A Small U-Net Class* slide: two levels down, a bottom, two levels up with transposed convolutions, and skip connections concatenated as channels.

## Task 4: Write the U-Net

Write `block(c_in, c_out)` returning two rounds of `Conv2d(3, padding=1), BatchNorm2d, ReLU` in one `nn.Sequential`, then `class UNetSmall(nn.Module)` for a 1-channel input: `self.down1 = block(1, 32)`, `self.down2 = block(32, 64)`, `self.bottom = block(64, 128)`, `self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)`, `self.dec2 = block(128, 64)`, `self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)`, `self.dec1 = block(64, 32)`, `self.out = nn.Conv2d(32, 1, 1)`. In `forward`: `d1 = down1(x)`; `d2 = down2(max_pool2d(d1, 2))`; `b = bottom(max_pool2d(d2, 2))`; `u2 = dec2(cat([up2(b), d2], dim=1))`; `u1 = dec1(cat([up1(u2), d1], dim=1))`; return `out(u1)`. Create `model = UNetSmall().to(device)`, print the output shape for `xb` (expected `(16, 1, 128, 128)`) and the parameter count.

Syntax hint (the shape of the call, not the answer):

```python
u = self.dec2(torch.cat([self.up2(b), d2], dim=1))   # upsample, then stack the encoder map as extra channels
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(out.shape) == (16, 1, 128, 128), 'UNetSmall: the output must have the input height and width; check the two cat calls and the final 1 x 1 conv'
print('Task 4 passed')

## Task 5: Dice, as a function

Write `dice_score(logits, masks)` that thresholds the logits at 0 (`pred = (logits > 0).float()`), computes per image `2 * (pred * masks).sum() / (pred.sum() + masks.sum() + 1e-6)` over the pixel axes `(1, 2, 3)`, and returns the mean over the batch as a Python number. Check it: `dice_score(torch.where(yb > 0, 10.0, -10.0), yb)` must be 1.0, and `dice_score(torch.full_like(yb, -10.0), yb)` close to 0.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(perfect - 1.0) < 1e-3 and nothing < 0.01, 'dice_score: 2 * overlap / (predicted + true pixels), per image, then the mean'
print('Task 5 passed')

---
# Part 3 -- Train, with Dice every epoch

The *Scoring a Mask* slide and Day 16's *Weights for Rare Classes*: about 2% of the pixels are tumour, so the positive class gets a weight of 10 in the loss, and the checkpoint follows Dice, not the loss.

## Task 6: The untrained network

Create `loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([10.0]).to(device))`. With `model.eval()` and `torch.no_grad()`, compute the mean loss and the mean `dice_score` over `val_loader` (weight each batch by its size) as `val_loss0, val_dice0` and print them. Dice should be low: the network has not seen a tumour yet.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert val_dice0 < 0.5, 'untrained: eval mode, no_grad, mean loss and Dice over the validation loader'
print('Task 6 passed')

## Task 7: The loop

Set `EPOCHS = 12` and `opt = torch.optim.Adam(model.parameters(), lr=1e-3)`. For each epoch: `model.train()`, loop over `train_loader` with the four moves, summing `loss.item() * len(xb)`; then `model.eval()` and the validation pass of Task 6 giving `val_loss` and `val_dice`; append `(train_loss, val_loss, val_dice)` to `history`; save `model.state_dict()` to `best_unet.pt` when `val_dice` improves; print the three numbers. Expected: Dice rising to about 0.6 by epoch 12 and still climbing; the checkpoint keeps the best epoch.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(history) == 12 and best_dice > 0.5 and os.path.exists('best_unet.pt'), 'loop: 12 epochs, Adam 1e-3, validation Dice per epoch, checkpoint on the best Dice'
print('Task 7 passed')

## Task 8: Read the curves

Plot the training and validation loss on the left and the validation Dice on the right (two subplots, markers, labels, titles). Which epoch did the checkpoint keep, and is the loss still falling there?

In [ ]:
# Your code here


---
# Part 4 -- Test once, look, hand in

## Task 9: Test with three metrics

Load `best_unet.pt` into a fresh `UNetSmall().to(device)` as `best`. Over `test_loader` (eval, no_grad) accumulate the pixel counts `TP` (predicted 1 and true 1), `FP`, `FN` and `TN` with `pred = logits > 0`, then print `dice = 2 TP / (2 TP + FP + FN)`, `iou = TP / (TP + FP + FN)`, `pixel accuracy = (TP + TN) / all`, and the accuracy of predicting background everywhere, `(TN + FP) / all`. Which of the four numbers would you report?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert test_dice > 0.5 and test_iou <= test_dice, 'test: counts over the test loader; Dice = 2TP / (2TP + FP + FN), IoU = TP / (TP + FP + FN)'
print('Task 9 passed')

## Task 10: Look at the masks

For the first 6 test slices show three rows: the slice, the true mask, and the predicted mask (`best(x[None].to(device))[0, 0] > 0`). Where does the network miss, and where does it add pixels?

In [ ]:
# Your code here


## Task 11: Hand in the masks

Predict every test slice, save each predicted mask as a PNG of 0 and 255 under `predicted_masks/` with the slice's name (`Image.fromarray((p.numpy() * 255).astype(np.uint8))`), and write `submission.csv` with one row per slice: `id` (the name) and `tumour_pixels` (the count of predicted pixels). Print the number of files written and the first 5 rows.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(os.listdir('predicted_masks')) == 200 and pd.read_csv('submission.csv').shape == (200, 2), 'hand-in: one PNG mask per test slice and a CSV with id and tumour_pixels'
print('Task 11 passed')

---
## Task 12: Reflect

Pixel accuracy was high even for a network that predicts no tumour at all. Explain in one sentence why Dice is the metric here, and what the positive weight of 10 in the loss did. The flip in `__getitem__` was applied to the image and the mask together. What would the network learn if only the image were flipped? The validation Dice was still rising at epoch 12. What is the obvious next move, and what would tell you to stop? Your U-Net has two levels; Ronneberger's has four: name one thing a deeper U-Net would see better.

*Your answers here*

---
## Conclusion

- **Masks as data**: image and mask resized and flipped together inside the Dataset, the mask kept at 0 and 1 with nearest-neighbour resizing.
- **The U-Net class**: blocks down with pooling, transposed convolutions up, skip connections concatenated as channels, a 1 x 1 convolution to one logit per pixel.
- **Training a rare class**: BCE with a positive weight, the checkpoint chosen by validation Dice, the curves read before testing.
- **Scoring and hand-in**: Dice and IoU against pixel accuracy on the test slices, the predicted masks looked at and saved as the submission.

---

Made By **Sattam Altwaim**